# 03 — Il confronto

Tutti i modelli finiti (feature, baseline, transformer con i loro seed), sugli stessi fold **per gruppo**. Le
metriche principali, fissate prima dei risultati: **F1 macro a 5 classi** e **accuratezza binaria**, per
testata. Si riportano anche accuratezza a 5 classi, tolerant 0.5 e 1, e il dettaglio per livello.

Gli intervalli al 95% vengono da un **bootstrap per gruppo** (2000 ricampionamenti dei gruppi di testate: le
testate dello stesso gruppo sbagliano insieme). Per ogni transformer, la **differenza** con il classificatore a
feature sugli stessi ricampionamenti.

Gira da solo alla fine di ogni esperimento della coda: si può guardare anche a metà weekend.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

SCHEMA = "fold_gruppo"
B = 2000

In [2]:
modelli = {}
for f in sorted(RESULTS.glob(f"feature*_{SCHEMA}_articoli.csv")):
    modelli[f.name.replace(f"_{SCHEMA}_articoli.csv", "")] = pd.read_csv(f)
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
print("modelli finiti:", list(modelli))
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo

modelli finiti: ['feature_1024tok', 'feature', 'modernbert_base_1024_s1', 'modernbert_base_1024_s2', 'modernbert_base_1024_s42', 'modernbert_base_2048_s1', 'modernbert_base_2048_s2', 'modernbert_base_2048_s42', 'modernbert_large_1024_s1', 'modernbert_large_1024_s2', 'modernbert_large_1024_s42']


In [3]:
PRINCIPALI = ["f1_macro", "bin_accuratezza", "accuratezza", "tolerant_0.5", "tolerant_1", "bin_f1_macro"]
righe, testate = {}, {}
for nome, p in modelli.items():
    for agg, fun in [("voto", per_testata), ("media", per_testata_media)]:
        t = fun(p); testate[(nome, agg)] = t
        righe[(nome, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
for f in sorted(RESULTS.glob(f"baseline_*_{SCHEMA}_metriche.json")):
    righe[(f.name.split(f"_{SCHEMA}")[0], "-")] = {k: v for k, v in json.load(open(f))["testata_voto"].items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T
tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[PRINCIPALI].round(3)

f1_macro  bin_accuratezza  \
modello                   aggregazione                              
feature_1024tok           voto             0.265            0.663   
                          media            0.252            0.648   
feature                   voto             0.274            0.653   
                          media            0.291            0.642   
modernbert_base_1024_s1   voto             0.231            0.648   
                          media            0.241            0.642   
modernbert_base_1024_s2   voto             0.281            0.642   
                          media            0.285            0.648   
modernbert_base_1024_s42  voto             0.240            0.622   
                          media            0.238            0.611   
modernbert_base_2048_s1   voto             0.256            0.653   
                          media            0.232            0.632   
modernbert_base_2048_s2   voto             0.290            0.632   
                          media            0.277            0.632   
modernbert_base_2048_s42  voto             0.207            0.606   
                          media            0.216            0.611   
modernbert_large_1024_s1  voto             0.271            0.617   
                          media            0.273            0.627   
modernbert_large_1024_s2  voto             0.250            0.601   
                          media            0.244            0.585   
modernbert_large_1024_s42 voto             0.228            0.596   
                          media            0.251            0.632   
baseline_a_caso           -                0.199            0.528   
baseline_maggioranza      -                0.097            0.606   

                                        accuratezza  tolerant_0.5  tolerant_1  \
modello                   aggregazione                                          
feature_1024tok           voto                0.311         0.521       0.731   
                          media               0.301         0.508       0.715   
feature                   voto                0.321         0.526       0.731   
                          media               0.326         0.521       0.715   
modernbert_base_1024_s1   voto                0.254         0.479       0.705   
                          media               0.275         0.487       0.699   
modernbert_base_1024_s2   voto                0.301         0.513       0.725   
                          media               0.306         0.510       0.715   
modernbert_base_1024_s42  voto                0.301         0.497       0.694   
                          media               0.301         0.487       0.674   
modernbert_base_2048_s1   voto                0.264         0.479       0.694   
                          media               0.233         0.451       0.668   
modernbert_base_2048_s2   voto                0.311         0.513       0.715   
                          media               0.301         0.508       0.715   
modernbert_base_2048_s42  voto                0.244         0.427       0.611   
                          media               0.249         0.433       0.617   
modernbert_large_1024_s1  voto                0.295         0.492       0.689   
                          media               0.295         0.495       0.694   
modernbert_large_1024_s2  voto                0.259         0.469       0.679   
                          media               0.254         0.461       0.668   
modernbert_large_1024_s42 voto                0.280         0.459       0.637   
                          media               0.316         0.492       0.668   
baseline_a_caso           -                   0.244         0.389       0.534   
baseline_maggioranza      -                   0.321         0.464       0.606   

                                        bin_f1_macro  
modello                   aggregazione                
feature_1024tok           vo

## Seed: media e deviazione standard

Gli esperimenti con lo stesso nome a parte il seed (`..._s42`, `..._s1`, `..._s2`).

In [4]:
x = tutto[PRINCIPALI].reset_index()
x["config"] = x.modello.str.replace(r"_s\d+$", "", regex=True)
seed = x.groupby(["config", "aggregazione"])[PRINCIPALI].agg(["mean", "std", "count"])
seed.to_csv(RESULTS / "03_confronto_seed.csv")
seed.round(3)

f1_macro              bin_accuratezza  \
                                       mean    std count            mean   
config                aggregazione                                         
baseline_a_caso       -               0.199    NaN     1           0.528   
baseline_maggioranza  -               0.097    NaN     1           0.606   
feature               media           0.291    NaN     1           0.642   
                      voto            0.274    NaN     1           0.653   
feature_1024tok       media           0.252    NaN     1           0.648   
                      voto            0.265    NaN     1           0.663   
modernbert_base_1024  media           0.255  0.026     3           0.634   
                      voto            0.251  0.027     3           0.637   
modernbert_base_2048  media           0.242  0.032     3           0.625   
                      voto            0.251  0.042     3           0.630   
modernbert_large_1024 media           0.256  0.015     3           0.615   
                      voto            0.250  0.022     3           0.604   

                                                accuratezza               \
                                      std count        mean    std count   
config                aggregazione                                         
baseline_a_caso       -               NaN     1       0.244    NaN     1   
baseline_maggioranza  -               NaN     1       0.321    NaN     1   
feature               media           NaN     1       0.326    NaN     1   
                      voto            NaN     1       0.321    NaN     1   
feature_1024tok       media           NaN     1       0.301    NaN     1   
                      voto            NaN     1       0.311    NaN     1   
modernbert_base_1024  media         0.020     3       0.294  0.017     3   
                      voto          0.014     3       0.285  0.027     3   
modernbert_base_2048  media         0.012     3       0.261  0.035     3   
                      voto          0.023     3       0.273  0.034     3   
modernbert_large_1024 media         0.026     3       0.288  0.032     3   
                      voto          0.011     3       0.278  0.018     3   

                                   tolerant_0.5              tolerant_1  \
                                           mean    std count       mean   
config                aggregazione                                        
baseline_a_caso       -                   0.389    NaN     1      0.534   
baseline_maggioranza  -                   0.464    NaN     1      0.606   
feature               media               0.521    NaN     1      0.715   
                      voto                0.526    NaN     1      0.731   
feature_1024tok       media               0.508    NaN     1      0.715   
                      voto                0.521    NaN     1      0.731   
modernbert_base_1024  media               0.495  0.013     3      0.696   
                      voto                0.497  0.017     3      0.708   
modernbert_base_2048  media               0.464  0.039     3      0.667   
                      voto                0.473  0.043     3      0.674   
modernbert_large_1024 media               0.483  0.019     3      0.677   
                      voto                0.473  0.017     3      0.668   

                                                bin_f1_macro               
                                      std count         mean    std count  
config                aggregazione                                         
baseline_a_caso       -               NaN     1        0.505    NaN     1  
baseline_maggioranza  -               NaN     1        0.377    NaN     1  
feature               media           NaN     1        0.628    NaN     1  
                      voto            NaN     1        0.639    NaN     1  
feature_1024tok       media           NaN     1        0.629    NaN     1  
                   

## Intervalli e differenze con il feature (bootstrap per gruppo)

In [5]:
rif = ("feature", "voto")
nomi = [k for k in testate if k[1] == "media" or k == rif]
dom = testate[rif].index
g = gruppo.reindex(dom).to_numpy()
gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx])
    return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(testate[k].reindex(dom), idx) for idx in campioni]) for k in nomi}
righe = {}
for k in nomi:
    b = boot[k]
    r = {"F1 macro": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1 vs feature": tutto.loc[k, "f1_macro"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "P(ΔF1>0)": (d[:, 0] > 0).mean(),
                  "Δbin vs feature": tutto.loc[k, "bin_accuratezza"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5), "P(Δbin>0)": (d[:, 1] > 0).mean()})
    righe[f"{k[0]} ({k[1]})"] = r
intervalli = pd.DataFrame(righe).T
intervalli.to_csv(RESULTS / "03_confronto_intervalli.csv")
intervalli.round(3)

,F1 macro,F1 2.5%,F1 97.5%,binario,bin 2.5%,bin 97.5%,ΔF1 vs feature,ΔF1 2.5%,ΔF1 97.5%,P(ΔF1>0),Δbin vs feature,Δbin 2.5%,Δbin 97.5%,P(Δbin>0)
feature_1024tok (media),0.252,0.166,0.330,0.648,0.560,0.739,-0.022,-0.063,0.020,0.167,-0.005,-0.048,0.032,0.360
feature (voto),0.274,0.177,0.355,0.653,0.562,0.747,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature (media),0.291,0.192,0.372,0.642,0.553,0.736,0.016,-0.007,0.049,0.891,-0.010,-0.036,0.015,0.158
modernbert_base_1024_s1 (media),0.241,0.179,0.300,0.642,0.562,0.728,-0.033,-0.136,0.079,0.298,-0.010,-0.088,0.061,0.362
modernbert_base_1024_s2 (media),0.285,0.193,0.361,0.648,0.560,0.739,0.011,-0.051,0.075,0.640,-0.005,-0.082,0.073,0.416
modernbert_base_1024_s42 (media),0.238,0.145,0.314,0.611,0.525,0.695,-0.037,-0.114,0.030,0.149,-0.041,-0.126,0.040,0.139
modernbert_base_2048_s1 (media),0.232,0.153,0.305,0.632,0.543,0.723,-0.042,-0.131,0.054,0.203,-0.021,-0.124,0.076,0.324
modernbert_base_2048_s2 (media),0.277,0.186,0.354,0.632,0.535,0.726,0.003,-0.079,0.081,0.544,-0.021,-0.102,0.054,0.278
modernbert_base_2048_s42 (media),0.216,0.159,0.278,0.611,0.526,0.705,-0.058,-0.157,0.052,0.180,-0.041,-0.156,0.071,0.224
modernbert_large_1024_s1 (media),0.273,0.187,0.350,0.627,0.543,0.714,-0.001,-0.090,0.094,0.526,-0.026,-0.134,0.071,0.294


## Per livello e matrici di confusione (testata, media delle probabilità; feature col voto)

In [6]:
LIV = ["l1", "l2", "l3", "l4", "l5"]
per_livello = pd.DataFrame({f"{k[0]} ({k[1]})": {f"{l} {q}": tutto.loc[k, f"{l}_{q}"] for l in LIV for q in ["recall", "f1"]} for k in nomi}).T
display(per_livello.round(2))
for k in nomi:
    t = testate[k]
    print(f"\n{k[0]} ({k[1]})")
    print(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]).to_string())

,l1 recall,l1 f1,l2 recall,l2 f1,l3 recall,l3 f1,l4 recall,l4 f1,l5 recall,l5 f1
feature_1024tok (media),0.26,0.29,0.33,0.32,0.07,0.09,0.47,0.42,0.15,0.15
feature (voto),0.26,0.32,0.47,0.41,0.11,0.13,0.42,0.40,0.11,0.11
feature (media),0.26,0.31,0.47,0.42,0.14,0.17,0.39,0.38,0.19,0.18
modernbert_base_1024_s1 (media),0.26,0.22,0.45,0.38,0.07,0.07,0.27,0.34,0.19,0.20
modernbert_base_1024_s2 (media),0.19,0.26,0.43,0.38,0.18,0.17,0.29,0.36,0.37,0.26
modernbert_base_1024_s42 (media),0.15,0.22,0.27,0.27,0.04,0.04,0.56,0.46,0.19,0.20
modernbert_base_2048_s1 (media),0.30,0.25,0.27,0.26,0.14,0.13,0.11,0.16,0.48,0.36
modernbert_base_2048_s2 (media),0.26,0.31,0.35,0.33,0.11,0.12,0.37,0.36,0.30,0.26
modernbert_base_2048_s42 (media),0.07,0.09,0.33,0.34,0.07,0.07,0.26,0.32,0.44,0.25
modernbert_large_1024_s1 (media),0.33,0.31,0.37,0.33,0.14,0.14,0.34,0.36,0.19,0.23



feature_1024tok (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1         7   6   5   9   0
l2        12  16   4  13   4
l3         1  13   2   9   3
l4         2  12   4  29  15
l5         0   5   1  17   4

feature (voto)
predetto  l1  l2  l3  l4  l5
vero                        
l1         7   7   6   7   0
l2         7  23   6   9   4
l3         1  13   3   8   3
l4         2  14   3  26  17
l5         0   5   1  18   3

feature (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1         7   6   5   9   0
l2         7  23   6   9   4
l3         1  12   4   8   3
l4         2  16   3  24  17
l5         1   4   1  16   5

modernbert_base_1024_s1 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1         7  14   3   2   1
l2        13  22   7   3   4
l3         9  11   2   4   2
l4         8  16  10  17  11
l5         1   4   5  12   5

modernbert_base_1024_s2 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1    